# Reddit scraper cho TIGER (Kaggle)

Notebook này chuyển quy trình trong `scrapper.py` sang các bước chạy tuần tự: lấy comment của subreddit, tạo interactions user–post, tiền xử lý theo định dạng JODIE/TGN và ghi dữ liệu để TIGER đọc.

## Trên Kaggle
1. Bật **Internet** trong Notebook settings.
2. Thêm Kaggle Notebook Secrets tên `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`, `REDDIT_USER_AGENT`. Không dán secret vào notebook.
3. Mở notebook trong repo hoặc clone repo rồi chạy notebook từ thư mục gốc `www2023tiger`.
4. Chạy từng cell từ trên xuống. Mặc định lấy 25.000 interactions; nếu Reddit không cung cấp đủ, cell thu thập sẽ báo lỗi.

Đầu ra: `data/reddit.csv`, `data/ml_reddit.csv`, `data/ml_reddit.npy`, `data/ml_reddit_node.npy`. Chỉ lưu ID số và features số; không lưu username hay nội dung comment. Hãy lưu output khỏi `/kaggle/working` trước khi runtime kết thúc.

In [ ]:
%pip install praw numpy pandas

In [ ]:
import csv
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import praw

TARGET_RECORDS = 25_000
SUBREDDIT_NAME = "MachineLearning"
MAX_SUBMISSIONS = 1_000  # Reddit listing giới hạn số submission gần đây.
MAX_MORE_PER_POST = None  # None: mở rộng toàn bộ comment MoreComments trả về.
OUTPUT_DIR = Path.cwd() / "data"

def get_credential(name):
    """Đọc credential từ Kaggle Secrets, hoặc biến môi trường khi chạy ngoài Kaggle."""
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError as exc:
        raise EnvironmentError(
            f"Thiếu {name}. Thêm vào Kaggle Notebook Secrets hoặc đặt biến môi trường."
        ) from exc
    value = UserSecretsClient().get_secret(name)
    if not value:
        raise EnvironmentError(f"Kaggle Secret {name} đang trống hoặc chưa được khai báo.")
    return value

if TARGET_RECORDS <= 0:
    raise ValueError("TARGET_RECORDS phải lớn hơn 0.")
if not 1 <= MAX_SUBMISSIONS <= 1_000:
    raise ValueError("MAX_SUBMISSIONS phải trong khoảng 1 đến 1000.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
reddit = praw.Reddit(
    client_id=get_credential("REDDIT_CLIENT_ID"),
    client_secret=get_credential("REDDIT_CLIENT_SECRET"),
    user_agent=get_credential("REDDIT_USER_AGENT"),
)
reddit.read_only = True
print(f"Reddit API đã cấu hình; output sẽ lưu tại {OUTPUT_DIR}")

## 1. Thu thập interactions

Mỗi comment hợp lệ tạo một event giữa tác giả comment và bài đăng chứa comment đó. ID tác giả chỉ được dùng để ánh xạ sang số nguyên trong bộ nhớ. Comment bị xóa, timestamp sai, comment trùng sẽ bị bỏ qua. `label=0` dành cho self-supervised link prediction, không phải nhãn phân loại.

In [ ]:
def collect_interactions(reddit_client, subreddit_name, target, max_submissions, max_more):
    user_ids = {}
    item_ids = {}
    seen_comment_ids = set()
    events = []
    subreddit = reddit_client.subreddit(subreddit_name)

    def get_user_id(author):
        if author is None:
            return None
        username = getattr(author, "name", None)
        if not username or username in {"[deleted]", "[removed]"}:
            return None
        if username not in user_ids:
            user_ids[username] = len(user_ids) + 1
        return user_ids[username]

    for submission in subreddit.new(limit=max_submissions):
        submission_id = submission.id
        if submission_id not in item_ids:
            item_ids[submission_id] = len(item_ids) + 1
        submission_time = float(submission.created_utc)
        submission.comments.replace_more(limit=max_more)

        for comment in submission.comments.list():
            comment_id = getattr(comment, "id", None)
            if not comment_id or comment_id in seen_comment_ids:
                continue
            seen_comment_ids.add(comment_id)
            user_id = get_user_id(getattr(comment, "author", None))
            if user_id is None:
                continue
            try:
                timestamp = float(comment.created_utc)
                score = max(0, int(comment.score))
                depth = max(0, int(comment.depth))
            except (AttributeError, TypeError, ValueError, OverflowError):
                continue
            if not math.isfinite(timestamp) or timestamp < submission_time:
                continue

            # Các feature số của interaction; không ghi text comment ra file.
            features = [
                math.log1p(score),
                float(depth),
                math.log1p(timestamp - submission_time),
                float(bool(getattr(comment, "is_submitter", False))),
            ]
            events.append({
                "user_id": user_id,
                "item_id": item_ids[submission_id],
                "timestamp": timestamp,
                "state_label": 0.0,
                "features": features,
            })
            if len(events) >= target:
                break
        if len(events) >= target:
            break

    if len(events) < target:
        raise RuntimeError(
            f"Chỉ thu được {len(events):,}/{target:,} interactions hợp lệ từ "
            f"{max_submissions:,} submissions gần đây của r/{subreddit_name}. "
            "Reddit giới hạn listing; hãy thử subreddit khác hoặc chạy lại sau."
        )

    # Đánh lại ID để bỏ khoảng trống do author/post không tạo event hợp lệ.
    used_users = sorted({event["user_id"] for event in events})
    used_items = sorted({event["item_id"] for event in events})
    user_reindex = {old_id: new_id for new_id, old_id in enumerate(used_users, 1)}
    item_reindex = {old_id: new_id for new_id, old_id in enumerate(used_items, 1)}
    for event in events:
        event["user_id"] = user_reindex[event["user_id"]]
        event["item_id"] = item_reindex[event["item_id"]]
    events.sort(key=lambda event: event["timestamp"])
    return events

print(f"Bắt đầu thu thập {TARGET_RECORDS:,} interactions từ r/{SUBREDDIT_NAME}...")
events = collect_interactions(
    reddit, SUBREDDIT_NAME, TARGET_RECORDS, MAX_SUBMISSIONS, MAX_MORE_PER_POST
)
print(f"Đã thu được {len(events):,} interactions.")

## 2. Tiền xử lý và lưu file

Tạo raw CSV theo cấu trúc JODIE (`user_id,item_id,timestamp,state_label,features`) và các file đã reindex theo quy ước bipartite của TGN để `load_jodie_data` trong TIGER đọc được.

In [ ]:
if len(events) != TARGET_RECORDS:
    raise RuntimeError(f"Cần đúng {TARGET_RECORDS:,} events, nhưng đang có {len(events):,}.")
feature_count = len(events[0]["features"])
if any(len(event["features"]) != feature_count for event in events):
    raise ValueError("Các interaction có số chiều feature không đồng nhất.")

raw_path = OUTPUT_DIR / "reddit.csv"
processed_path = OUTPUT_DIR / "ml_reddit.csv"
edge_features_path = OUTPUT_DIR / "ml_reddit.npy"
node_features_path = OUTPUT_DIR / "ml_reddit_node.npy"

with raw_path.open("w", newline="", encoding="utf-8") as output:
    writer = csv.writer(output)
    writer.writerow([
        "user_id", "item_id", "timestamp", "state_label",
        "comma_separated_list_of_features",
    ])
    for event in events:
        writer.writerow([
            event["user_id"], event["item_id"], event["timestamp"],
            event["state_label"], *event["features"],
        ])

processed = pd.DataFrame({
    "u": [event["user_id"] for event in events],
    "i": [event["item_id"] for event in events],
    "ts": [event["timestamp"] for event in events],
    "label": [event["state_label"] for event in events],
    "idx": np.arange(len(events), dtype=np.int64),
})

# TGN bipartite reindexing; reserve node/edge ID 0 for padding.
max_user_id = int(processed["u"].max())
processed["i"] = processed["i"] + max_user_id + 1
processed["u"] = processed["u"] + 1
processed["i"] = processed["i"] + 1
processed["idx"] = processed["idx"] + 1
processed = processed[["u", "i", "ts", "label", "idx"]]
processed.to_csv(processed_path, index=False)

edge_features = np.asarray([event["features"] for event in events], dtype=np.float32)
edge_features = np.vstack([
    np.zeros((1, feature_count), dtype=np.float32), edge_features
])
max_node_id = int(max(processed["u"].max(), processed["i"].max()))
node_features = np.zeros((max_node_id + 1, feature_count), dtype=np.float32)
np.save(edge_features_path, edge_features)
np.save(node_features_path, node_features)

print(f"Đã lưu raw interactions: {raw_path}")
print(f"Đã lưu TIGER dataset: {processed_path}")
print(f"Đã lưu edge features: {edge_features_path}")
print(f"Đã lưu node features: {node_features_path}")

## 3. Kiểm tra dữ liệu đầu ra

Xác nhận số dòng, schema, thứ tự thời gian và kích thước feature trước khi dùng TIGER. Cell này không bắt đầu train model.

In [ ]:
check = pd.read_csv(processed_path)
loaded_edge_features = np.load(edge_features_path, mmap_mode="r")
loaded_node_features = np.load(node_features_path, mmap_mode="r")

if len(check) != TARGET_RECORDS:
    raise RuntimeError(f"CSV có {len(check):,} dòng, mong đợi {TARGET_RECORDS:,}.")
if list(check.columns) != ["u", "i", "ts", "label", "idx"]:
    raise ValueError(f"Schema CSV sai: {list(check.columns)}")
if not check["ts"].is_monotonic_increasing:
    raise ValueError("Timestamp chưa được sắp xếp tăng dần.")
if loaded_edge_features.shape != (TARGET_RECORDS + 1, feature_count):
    raise ValueError(f"Kích thước edge features sai: {loaded_edge_features.shape}")
if loaded_node_features.shape != (max_node_id + 1, feature_count):
    raise ValueError(f"Kích thước node features sai: {loaded_node_features.shape}")

print("Kiểm tra thành công.")
print(f"Interactions: {len(check):,} | Nodes: {max_node_id:,} | Feature dim: {feature_count}")